# PCA90 and Manifold Alignment: A Visual Walkthrough

This notebook starts from edited motor-unit discharge times and reproduces the tracked-MU PCA/CCA workflow.
All explanations, tables, plot labels and exported filenames are in English.

The displayed results are limited to:

1. **A PCA90 table:** how many principal components explain at least 90% of the variance in each condition.
2. **Before/after alignment images:** source and target trajectories in 2D and 3D.
3. **Control images:** temporal-block shuffling and whole-space CCA fitted with incorrect task correspondences.
4. **Alignment metric tables:** mode-by-mode canonical correlations and their top-three mean for observed data and controls.

The default example is **sub1, wrist, 0° → 10°**. Run the cells from top to bottom.
No classification, numerical validation tables, correlation heatmaps or statistical comparison plots are displayed.
Input paths are intentionally blank. Fill in the configuration cell before running the analysis.
Results are displayed inline by default; exporting separate files is optional and disabled.

## 1. Select the data and comparison

A condition is specified as `(subject, location, angle)`.
For example, use `('sub1', 'wrist', 0)` and `('sub2', 'wrist', 0)` for a cross-subject comparison,
or `('sub1', 'wrist', 0)` and `('sub1', 'forearm', 0)` for a cross-location comparison.

The PCA90 table is recomputed from every configured condition (20 with the default settings).
Only the selected source and target are used in the trajectory images.

### Required input files

Fill in `DATA_ROOT` and `DICTIONARY_ROOT` in the configuration cell. Both are intentionally empty in the distributed notebook. Absolute paths or paths relative to the notebook's working directory are accepted. On Windows, use forward slashes or a raw string when entering a path.

#### 1. Edited motor-unit recordings (`DATA_ROOT`)

The input directory must contain the following structure (the subject below is an example):

```text
<DATA_ROOT>/
  wrist_muedit_new/
    sub1/
      sub1_staircase_flx_1_index_15MVC_wrist_decomp_0deg_muedit.mat_edited.mat
      sub1_staircase_flx_1_middle_15MVC_wrist_decomp_0deg_muedit.mat_edited.mat
      sub1_staircase_flx_1_index-middle_15MVC_wrist_decomp_0deg_muedit.mat_edited.mat
      ...
  forearm_muedit_new/
    sub1/
      sub1_staircase_flx_1_index_15MVC_forearm_decomp_0deg_muedit.mat_edited.mat
      ...
```

Use your actual subject names, such as `sub1`, in both the folders and filenames. The current reader expects this naming pattern; task names are exactly `index`, `middle`, and `index-middle`. Angles are nonnegative integers followed by `deg`. Each configured subject and angle needs one recording for each of the three tasks at **both** locations, wrist and forearm.

With the default `SUBJECTS = ('sub1', 'sub2')` and `ANGLES = (0, 10, 20, 30, 40)`, this means **60 recordings and 20 condition dictionaries**. Edit `SUBJECTS`, `ANGLES`, `SOURCE`, and `TARGET` to match your dataset. Source and target must belong to the configured cohort. The current reader always includes both locations.

Files must be **MATLAB v7.3 / HDF5** files with these fields:

| MATLAB field | Required content | Use in this notebook |
| --- | --- | --- |
| `signal.fs` (or `signal.fsamp`) | A finite, positive sampling rate in Hz | Convert discharge events into firing-rate signals; all recordings must have the same sampling rate |
| `edition.Pulsetrainclean{1}` | A two-dimensional matrix, MUs × samples in MATLAB | Determine the number of MUs and samples |
| `edition.Distimeclean{1}` | A cell array with one discharge-event vector per MU, in the same MU order as the pulse matrix | Reconstruct binary spike trains |

Discharge events must be **one-based sample indices**, not times in seconds. Empty event vectors are allowed. The reader uses the **first grid only** (`{1}`), and handles MATLAB/HDF5 dimension reversal internally. Other MAT formats require conversion to v7.3 before loading. Raw EMG, force traces, `edition.time`, MUAP templates, and precomputed PCA or alignment results are not required by this notebook. MATLAB is not needed to run the Python analysis once these inputs are available.

#### 2. Existing MU tracking dictionaries (`DICTIONARY_ROOT`)

Place one CSV per subject, location and angle directly in the dictionary directory:

```text
<DICTIONARY_ROOT>/
  unique_mu_dictionary_sub1_wrist_0deg.csv
  unique_mu_dictionary_sub1_forearm_0deg.csv
  ...
```

Required CSV columns:

| Column | Meaning |
| --- | --- |
| `unique_mu` | Stable MU identity shared by all tracked appearances of that MU across tasks within this condition |
| `task` | Exactly `index`, `middle`, or `index-middle` |
| `mu_index0` | Zero-based MU row index in that task's edited pulse matrix; this must refer to the current edited file |

For example, the following illustrates the schema only; it is not a complete dataset:

```csv
unique_mu,task,mu_index0
UMU001,index,0
UMU001,middle,2
UMU002,index-middle,0
```

Here, row 1 of the Index recording and row 3 of the Middle recording are tracked as the same MU, `UMU001`. Their activity is placed in the **same feature column**, in their respective task time blocks, before PCA. `UMU002` occupies a separate column. If an identity has no entry for a task, that task's block for that column is filled with zero.

The notebook **consumes existing tracking results; it does not perform MU tracking or infer which units match**. Identity scope is one subject × location × angle across the three tasks. Dictionaries do not establish shared identities across subjects, locations or angles.

Every task must have dictionary entries. Required values must be nonempty; indices must be integers within the corresponding recording's MU range. Each `(unique_mu, task)` and each `(task, mu_index0)` combination must be unique. Give untracked MUs distinct identities and include every MU that should participate in PCA: MUs omitted from the dictionary are excluded. If optional `subject`, `location`, or `angle_deg` columns are present, they must match the dictionary filename. Additional tracking-quality columns are allowed but do not affect the analysis.

#### 3. Optional output directory

`SAVE_OUTPUTS = False` and `OUTPUT_DIR = ""` are the defaults. Results appear inline; the analysis does not export separate files or create an output folder. Jupyter may still save the notebook itself and its displayed outputs.

To export results, set `SAVE_OUTPUTS = True` and enter a directory in `OUTPUT_DIR`. The notebook then writes PNG figures, CSV tables, an NPZ audit file and a JSON settings record there. The settings record includes the input paths you configured. Before committing a notebook that you have run locally, clear its outputs and reset its path settings to empty strings if you want to keep the published copy portable.

In [ ]:
from pathlib import Path
import os, json
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(min(4, os.cpu_count() or 1)))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, HTML
from manifold_workflow import (
    TASKS, inventory, load_features, pca_fit,
    cca_fit, cca_transform, block_shuffle,
)

DATA_ROOT = ""         # Folder containing wrist_muedit_new/ and forearm_muedit_new/.
DICTIONARY_ROOT = ""   # Folder containing unique_mu_dictionary_*.csv.
SAVE_OUTPUTS = False   # False: display results inline without writing output files.
OUTPUT_DIR = ""        # Required only when SAVE_OUTPUTS is True; no default output folder.
SUBJECTS = ('sub1', 'sub2')
ANGLES = (0, 10, 20, 30, 40)
SOURCE = ('sub1', 'wrist', 0)
TARGET = ('sub1', 'wrist', 10)
SEED = 1
DISPLAY_STRIDE = 20  # Display only; PCA and CCA use every native sample.
COLORS = ['#0072B2', '#D55E00', '#009E73']
TASK_LABELS = ['Index', 'Middle', 'Index–middle']
plt.rcParams.update({'figure.dpi': 110, 'font.family':'DejaVu Sans',
                     'font.size':10, 'axes.spines.top':False, 'axes.spines.right':False})

def require_input_directory(value, setting):
    if not str(value).strip():
        raise ValueError(f'Please fill in {setting}. See Required input files above.')
    path = Path(value).expanduser()
    if not path.is_dir():
        raise ValueError(f'{setting} must point to an existing input directory.')
    return path

EXPORT_ROOT = None
if SAVE_OUTPUTS:
    if not str(OUTPUT_DIR).strip():
        raise ValueError('Fill in OUTPUT_DIR to enable exports, or set SAVE_OUTPUTS = False.')
    EXPORT_ROOT = Path(OUTPUT_DIR).expanduser()
    EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def export_table(table, filename):
    if SAVE_OUTPUTS:
        table.to_csv(EXPORT_ROOT/filename, index=False)

## 2. Prepare the tracked-MU activity matrices

The preprocessing matches the inspected `new_manifold` pipeline:

**Edited discharge times → binary spike trains → 400 ms periodic Hann smoothing →
0.75 Hz high-pass filtering → per-MU min–max normalization → task-tracked MU matrix.**

The three tasks are cropped from their starts to their task-specific minimum lengths across the cohort,
then stacked in the order Index, Middle, Index–middle. Columns represent unique MUs in the tracking dictionary;
task blocks without a dictionary member are filled with zero, as in the original analysis.

Each condition is mean-centered and fitted with PCA. The table reports the minimum number of PCs needed
for 90% variance. The original workflow retains at least three dimensions for downstream analysis, even when two
already explain 90%; the last column makes this distinction explicit.

In [ ]:
data_root = require_input_directory(DATA_ROOT, 'DATA_ROOT')
dictionary_root = require_input_directory(DICTIONARY_ROOT, 'DICTIONARY_ROOT')
manifest = inventory(data_root, subjects=SUBJECTS, angles=ANGLES)
crop_counts = manifest.groupby('task').n_samples.min().to_dict()
selected = {}
pca_rows = []
for subject in SUBJECTS:
    for location in ['wrist', 'forearm']:
        for angle in ANGLES:
            key = (subject, location, angle)
            feature = load_features(manifest, dictionary_root, key, crop_counts)
            model = pca_fit(feature['F'])
            pca_rows.append({
                'Subject':subject, 'Location':location.capitalize(), 'Angle (deg)':angle,
                'Unique MUs':feature['F'].shape[1], 'PCs for 90% variance':model['k90'],
                'Variance explained (%)':np.sum(model['explained'][:model['k90']]),
                'Dimensions retained for alignment':model['X'].shape[1],
            })
            if key in [SOURCE, TARGET]:
                selected[key] = {'F':feature['F'], 'y':feature['y'], 'pca':model}
if SOURCE not in selected or TARGET not in selected:
    raise ValueError('SOURCE and TARGET must belong to the selected cohort.')
source, target = selected[SOURCE], selected[TARGET]
assert np.array_equal(source['y'], target['y']), 'Source and target must have matching task rows.'
y = source['y']
pca_table = pd.DataFrame(pca_rows)
display(pca_table.style.hide(axis='index').format({'Variance explained (%)':'{:.2f}'}))
export_table(pca_table, 'pca90_dimensionality.csv')

## 3. Define the visual conventions

**Color identifies the task. Solid lines show the source; dashed lines show the target.**
Open circles mark the beginning of a task trajectory. The images show the first two or three modes;
CCA is fitted using all retained dimensions.

Before alignment, source and target have independent PCA coordinate systems.
After alignment, target canonical scores are weighted by their canonical correlations and scaled using the source,
following the original MATLAB implementation.

The observed and shuffled CCA fits each define their own axes. Compare source–target overlap **within** a panel;
do not interpret a rotation between panels as a change in the underlying activity by itself.

In [ ]:
def condition_label(key):
    return f'{key[0]} / {key[1]} / {key[2]}°'

def alignment_metrics(label, mapping, S, T):
    r = np.asarray(mapping['r'])
    row = {'Analysis':label, 'Paired samples':len(S),
           'Source PCs':S.shape[1], 'Target PCs':T.shape[1], 'CCA modes':len(r)}
    row.update({f'r{mode+1}':float(value) for mode,value in enumerate(r)})
    row['Mean top-3 r'] = float(np.mean(r[:3]))
    return row

def display_metrics(table):
    formats = {column:'{:.4f}' for column in table.columns
               if column.startswith('r') or column == 'Mean top-3 r'}
    display(table.style.hide(axis='index').format(formats, na_rep='—'))

def padded(values):
    return np.pad(values, ((0,0), (0,max(0,3-values.shape[1]))))

def limits_for(arrays, dims):
    values = np.vstack([padded(a)[:, :dims] for a in arrays])
    lo, hi = values.min(axis=0), values.max(axis=0)
    margin = np.maximum(hi-lo, 1e-3)*.08
    return list(zip(lo-margin, hi+margin))

def draw_trajectories(ax, S, T, labels, dims, aligned=False, limits=None, task_ids=(1,2,3)):
    S, T = padded(S), padded(T)
    for task_id in task_ids:
        idx = np.flatnonzero(labels == task_id)
        draw = np.unique(np.r_[idx[::DISPLAY_STRIDE], idx[-1]])
        for values, style in [(S, '-'), (T, '--')]:
            args = [values[draw,j] for j in range(dims)]
            ax.plot(*args, color=COLORS[task_id-1], linestyle=style, linewidth=1.15, alpha=.9)
            start = [values[idx[0],j:j+1] for j in range(dims)]
            ax.plot(*start, marker='o', markersize=4, markerfacecolor='white',
                    markeredgecolor=COLORS[task_id-1], linestyle='None')
    prefix = 'Canonical mode' if aligned else 'PC'
    ax.set_xlabel(f'{prefix} 1')
    ax.set_ylabel(f'{prefix} 2')
    if dims == 3:
        short = 'CC' if aligned else 'PC'
        ax.set_xlabel(f'{short} 1', fontsize=9, labelpad=2)
        ax.set_ylabel(f'{short} 2', fontsize=9, labelpad=2)
        ax.set_zlabel(f'{short} 3', fontsize=9, labelpad=2)
        ax.view_init(elev=23, azim=-55)
        ax.set_box_aspect((1,1,1), zoom=.82)
        from matplotlib.ticker import MaxNLocator
        for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
            axis.set_major_locator(MaxNLocator(nbins=4))
        ax.tick_params(labelsize=8)
    else:
        ax.set_aspect('equal', adjustable='box')
    if limits is not None:
        ax.set_xlim(limits[0]); ax.set_ylim(limits[1])
        if dims == 3: ax.set_zlim(limits[2])

def add_legend(fig):
    handles = [Line2D([0],[0], color=c, lw=2, label=t) for c,t in zip(COLORS,TASK_LABELS)]
    handles += [Line2D([0],[0], color='#333333', lw=1.4, label='Source', linestyle='-'),
                Line2D([0],[0], color='#333333', lw=1.4, label='Target', linestyle='--')]
    fig.legend(handles=handles, loc='lower center', ncol=5, frameon=False,
               bbox_to_anchor=(.5,.01), fontsize=10)

def save_image(fig, name):
    if SAVE_OUTPUTS:
        fig.savefig(EXPORT_ROOT/f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

Xs, Xt = source['pca']['X'], target['pca']['X']
observed_mapping = cca_fit(Xs, Xt)
observed_source, observed_target = cca_transform(observed_mapping, Xs, Xt)

## 4. Observed trajectories: before and after CCA

Read each row from left to right. The left panel shows independent PCA trajectories;
the right panel shows the source and target after CCA alignment.

The accompanying table reports **r1, r2, r3, ...** for every fitted canonical mode and
**Mean top-3 r**, the summary used in the original analysis. It also lists the paired sample count,
source/target PCA dimensions and the number of canonical modes.
These r values are the regularized CCA singular values returned by the original algorithm;
they can differ slightly from direct Pearson correlations of the displayed canonical scores because of regularization.
All metrics use the full paired data, not just the points subsampled for display. They describe the fitted alignment,
not held-out performance. PCA explained variance and canonical correlation are different quantities.

In [ ]:
observed_metrics = pd.DataFrame([alignment_metrics('Observed', observed_mapping, Xs, Xt)])
display_metrics(observed_metrics)
export_table(observed_metrics, 'observed_alignment_metrics.csv')

for dims in [2,3]:
    fig = plt.figure(figsize=(13,5.6))
    fig.suptitle(f'Observed activity: {condition_label(SOURCE)} → {condition_label(TARGET)}', y=.97)
    for col,S,T,aligned,title in [(1,Xs,Xt,False,'Before alignment'),
                                 (2,observed_source,observed_target,True,'After CCA alignment')]:
        ax = fig.add_subplot(1,2,col,projection='3d' if dims==3 else None)
        draw_trajectories(ax,S,T,y,dims,aligned=aligned,limits=limits_for([S,T],dims))
        ax.set_title(title, pad=14)
    fig.subplots_adjust(left=.07,right=.93,bottom=.18,top=.85,wspace=.26)
    add_legend(fig)
    save_image(fig,f'observed_before_after_{dims}d')

## 5. Temporal-block control: observed versus shuffled activity

Each task is divided into non-overlapping blocks of length `floor(10% × task samples)`.
All MU columns in a task share one block permutation. Different tasks, source and target receive independent
permutations; any incomplete tail remains in place. PCA and CCA are then recomputed.

This control preserves task labels, task means, missing-MU blocks and the within-task population sample distribution.
It disrupts long-range temporal correspondence. Consequently, visible task clusters can remain after shuffling.
Line jumps in the control images represent transitions between rearranged time blocks.

The **top row is observed data** and the **bottom row is the shuffled control**.
Columns show trajectories before and after alignment. Limits are shared within each column.
PCA loading signs are matched to their observed counterparts only to remove arbitrary SVD sign flips.

The table below compares the **observed fit** with the **same shuffled realization shown in the images**.
The shuffled row reports correlations after refitting PCA and CCA. This is one realization with the configured seed,
not an average across repetitions or a significance test.

In [ ]:
rng = np.random.default_rng(SEED)
Fs_control, source_permutations = block_shuffle(source['F'], y, rng)
Ft_control, target_permutations = block_shuffle(target['F'], y, rng)
sp_control, tp_control = pca_fit(Fs_control), pca_fit(Ft_control)
for control_model, observed_model in [(sp_control,source['pca']), (tp_control,target['pca'])]:
    n = min(control_model['X'].shape[1], control_model['coeff'].shape[1])
    signs = np.sign(np.sum(control_model['coeff'][:,:n]*observed_model['coeff'][:,:n],axis=0))
    signs[signs==0] = 1
    control_model['X'][:,:n] *= signs
Xsc, Xtc = sp_control['X'], tp_control['X']
control_mapping = cca_fit(Xsc, Xtc)
control_source, control_target = cca_transform(control_mapping, Xsc, Xtc)

alignment_control_metrics = pd.DataFrame([
    alignment_metrics('Observed', observed_mapping, Xs, Xt),
    alignment_metrics('Temporal-block shuffle', control_mapping, Xsc, Xtc),
])
display_metrics(alignment_control_metrics)
export_table(alignment_control_metrics, 'observed_vs_shuffle_alignment_metrics.csv')

for dims in [2,3]:
    fig = plt.figure(figsize=(13,10))
    fig.suptitle('Temporal-block control: observed and shuffled trajectories', y=.985)
    before_limits = limits_for([Xs,Xt,Xsc,Xtc],dims)
    after_limits = limits_for([observed_source,observed_target,control_source,control_target],dims)
    panels = [
        (1,Xs,Xt,False,'Observed — before alignment',before_limits),
        (2,observed_source,observed_target,True,'Observed — after CCA',after_limits),
        (3,Xsc,Xtc,False,'Shuffled — before alignment',before_limits),
        (4,control_source,control_target,True,'Shuffled — after CCA',after_limits),
    ]
    for panel,S,T,aligned,title,limits in panels:
        ax = fig.add_subplot(2,2,panel,projection='3d' if dims==3 else None)
        draw_trajectories(ax,S,T,y,dims,aligned=aligned,limits=limits)
        ax.set_title(title,pad=12)
    fig.subplots_adjust(left=.08,right=.92,bottom=.1,top=.93,wspace=.28,hspace=.32)
    add_legend(fig)
    save_image(fig,f'temporal_control_before_after_{dims}d')

## 6. Whole-space task-mismatch control

**Keep the entire pooled latent space. Change only the task correspondence used to fit CCA.**
The original pooled PCA bases and retained dimensions stay fixed. Every CCA fit includes all three tasks together.
Each fit produces one source projection and one target projection; each dataset's projection is shared by all its
task trajectories. No task-specific CCA projections are fitted.

For example, one incorrect correspondence is:

**Source Index ↔ Target Middle; Source Middle ↔ Target Index–middle;
Source Index–middle ↔ Target Index.**

Both possible cyclic permutations that mismatch all three tasks are shown. The mapping table specifies the
target task assigned to each source task during fitting.
After fitting, apply the global projections to the original, complete source and target trajectories.
**Plot colors always retain the true task identities.** Thus a blue source trajectory overlapping an orange target
trajectory illustrates wrong-task alignment; the target is never recolored to hide the mismatch.

The native task lengths differ. To isolate task correspondence from length differences, use the first
`min(task lengths)` samples of each task for all three CCA fits in this section: the correct-correspondence baseline
and both mismatched controls. This selects the same sample sets for every fit, merely reordered on the target side.
The PCA bases remain those fitted to the full pooled datasets, and the plots show **all original rows** after projection.
Consequently, this section's correct baseline uses slightly fewer fitting rows than Section 4; compare each control
against the matched-length correct baseline in this section.

The canonical correlations in the table are calculated under the correspondence used to fit that model.
A high value for an incorrect mapping does not mean true task identities have been aligned correctly.
This control evaluates a single global alignment under each pooled task correspondence.

In [ ]:
def pooled_pairing_indices(labels, target_order):
    # One row pairing for a single global CCA fit spanning all three tasks.
    if sorted(target_order) != [1,2,3]:
        raise ValueError('Target task order must be a permutation of (1, 2, 3).')
    rows_by_task = {c:np.flatnonzero(labels==c) for c in [1,2,3]}
    count = min(len(rows) for rows in rows_by_task.values())
    source_rows = np.concatenate([rows_by_task[c][:count] for c in [1,2,3]])
    target_rows = np.concatenate([rows_by_task[c][:count] for c in target_order])
    # Every fit uses the same samples; only their target-side correspondence changes.
    assert np.array_equal(np.sort(source_rows), np.sort(target_rows))
    return source_rows, target_rows, count

correspondences = [
    ('Correct correspondence', (1,2,3)),
    ('Wrong correspondence A', (2,3,1)),
    ('Wrong correspondence B', (3,1,2)),
]
pairing_table = pd.DataFrame([
    {'Analysis':label, **{f'Source {TASK_LABELS[i]} paired with':TASK_LABELS[order[i]-1]
                         for i in range(3)}}
    for label,order in correspondences
])
display(pairing_table.style.hide(axis='index'))
export_table(pairing_table, 'whole_space_task_correspondences.csv')

whole_space_fits = []
whole_space_metric_rows = []
for label,order in correspondences:
    source_fit_rows, target_fit_rows, samples_per_task = pooled_pairing_indices(y,order)
    Sfit, Tfit = Xs[source_fit_rows], Xt[target_fit_rows]
    # Exactly one CCA model for the entire stacked latent space under this correspondence.
    mapping = cca_fit(Sfit, Tfit)
    # Project the full original arrays; do not plot target rows in their fitting permutation.
    Sfull, Tfull = cca_transform(mapping, Xs, Xt)
    whole_space_fits.append({'label':label, 'order':order, 'mapping':mapping,
                             'source':Sfull, 'target':Tfull,
                             'source_fit_rows':source_fit_rows, 'target_fit_rows':target_fit_rows})
    whole_space_metric_rows.append(alignment_metrics(label,mapping,Sfit,Tfit))

whole_space_metrics = pd.DataFrame(whole_space_metric_rows)
display_metrics(whole_space_metrics)
export_table(whole_space_metrics, 'whole_space_task_mismatch_metrics.csv')

for dims in [2,3]:
    fig = plt.figure(figsize=(13,10))
    fig.suptitle('Whole latent space: correct versus incorrect task correspondence', y=.985)
    after_limits = limits_for([values for fit in whole_space_fits
                              for values in [fit['source'],fit['target']]],dims)
    panels = [(1,Xs,Xt,False,'Before alignment — all three tasks',limits_for([Xs,Xt],dims))]
    panels += [(i+2,fit['source'],fit['target'],True,fit['label']+' — global CCA',after_limits)
               for i,fit in enumerate(whole_space_fits)]
    for panel,S,T,aligned,title,limits in panels:
        ax = fig.add_subplot(2,2,panel,projection='3d' if dims==3 else None)
        # Same y and same task colors for source/target in every panel: true identities.
        draw_trajectories(ax,S,T,y,dims,aligned=aligned,limits=limits)
        ax.set_title(title,pad=12,fontsize=11)
    fig.subplots_adjust(left=.08,right=.92,bottom=.1,top=.93,wspace=.28,hspace=.32)
    add_legend(fig)
    save_image(fig,f'whole_space_task_mismatch_{dims}d')

# Compact audit data: one projection pair per correspondence, plus the complete plotted trajectories.
audit_arrays = {'source_pca':Xs, 'target_pca':Xt, 'true_labels':y}
for i,fit in enumerate(whole_space_fits):
    for key in ['source','target','source_fit_rows','target_fit_rows']:
        audit_arrays[f'fit{i}_{key}'] = fit[key]
    for key,value in fit['mapping'].items():
        audit_arrays[f'fit{i}_map_{key}'] = value
if SAVE_OUTPUTS:
    np.savez_compressed(EXPORT_ROOT/'whole_space_task_mismatch_audit.npz',**audit_arrays)

## 7. Complete the analysis and optionally export

By default, all tables and figures are displayed in the notebook and no separate output files are written.
To export PNG images, CSV tables and reproducibility records, explicitly set `SAVE_OUTPUTS = True`, fill in
`OUTPUT_DIR`, and rerun the notebook. There is no automatic output directory.
The plotted trajectories are subsampled only for readability; PCA uses all native rows retained after cropping.
The whole-space task-mismatch fits use the matched-length subsets described in Section 6.

These images show the geometry of the fitted representations. Visual overlap alone does not establish
independent-trial generalization or prove that detailed neural dynamics are preserved.

In [ ]:
settings = {
    'source':SOURCE, 'target':TARGET, 'subjects':SUBJECTS, 'angles':ANGLES,
    'data_root':str(DATA_ROOT), 'dictionary_root':str(DICTIONARY_ROOT),
    'native_crop_counts':crop_counts, 'seed':SEED, 'display_stride':DISPLAY_STRIDE,
    'preprocessing':'400 ms periodic Hann; 0.75 Hz high-pass; min-max per MU',
    'pca':'Mean-centered; retain max(3, PCs explaining 90% variance)',
    'control':'Independent taskwise 10% temporal-block permutations for source and target',
    'source_permutations':source_permutations, 'target_permutations':target_permutations,
    'whole_space_task_correspondences':{label:order for label,order in correspondences},
    'whole_space_fit_samples_per_task':int(samples_per_task),
    'whole_space_plot_labels':'Original true task identities; all native rows projected',
}
if SAVE_OUTPUTS:
    (EXPORT_ROOT/'visualization_settings.json').write_text(json.dumps(settings,indent=2),encoding='utf-8')
    print('Analysis complete. Files were exported to the configured output directory.')
else:
    print('Analysis complete. Tables and figures are displayed above; no output files were exported.')